# Question 2 - 27. Remove Element

Given an integer array `nums` and an integer `val`, remove all occurrences of `val` in `nums` **in-place**. The order of the elements may be changed. Then return *the number of elements in* `nums` *which are not equal to* `val`.

Consider the number of elements in `nums` which are not equal to `val` be `k`, to get accepted, you need to do the following things:

- Change the array `nums` such that the first `k` elements of `nums` contain the elements which are not equal to `val`. The remaining elements of `nums` are not important as well as the size of `nums`.
- Return `k`.

**Custom Judge:**

    int[] nums = [...]; // Input array
    int val = ...; // Value to remove
    int[] expectedNums = [...]; // The expected answer with correct length.
                                // It is sorted with no values equaling val.

    int k = removeElement(nums, val); // Calls your implementation

    assert k == expectedNums.length;
    sort(nums, 0, k); // Sort the first k elements of nums
    for (int i = 0; i < actualLength; i++) {
        assert nums[i] == expectedNums[i];
    }

If all assertions pass, then your solution will be accepted.

**Example 1:**

    Input: nums = [3,2,2,3], val = 3
    Output: 2, nums = [2,2,_,_]
    Explanation: Your function should return k = 2, with the first two elements of nums being 2.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Example 2:**

    Input: nums = [0,1,2,2,3,0,4,2], val = 2
    Output: 5, nums = [0,1,4,0,3,_,_,_]
    Explanation: Your function should return k = 5, with the first five elements of nums containing 0, 0, 1, 3, and 4.
    Note that the five elements can be returned in any order.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Constraints:**

- `0 <= nums.length <= 100`
- `0 <= nums[i] <= 50`
- `0 <= val <= 100`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers: a read pointer and a write pointer (in-place compaction)

📘 **Revise first:** [Part 3.5 · why removing is O(n)](./0-concepts-array-and-string.ipynb) · [Pattern A · read/write pointers](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (delete one at a time) | O(n²) | O(1) |
| 2️⃣ Optimized (read/write pointers) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

You have a row of boxes with numbers in them. Someone says: *"Get rid of every box labelled `val`."*

But the boxes are **bolted to the floor**. You can't remove a box or create new ones. All you can do is **change what's written inside** each box. So the real task is:

> Rearrange the contents so that **all the "good" numbers (≠ `val`) are packed at the front**, and tell me **how many** good numbers there are (`k`).

Whatever is left in the boxes after position `k` is junk. The judge ignores it.

```
nums = [3, 2, 2, 3],  val = 3
goal → [2, 2, ?, ?]   and return k = 2
```

**Everyday picture:** you're tidying a bookshelf and want to remove all the comics, keeping the novels packed to the left. You walk along the shelf once with your eyes (the **reader**). Every time you spot a novel, you place it at the next free spot on the left (the **writer**). Comics are simply skipped. At the end, the novels are packed at the front and you know how many there are.

### Step 0 · Clarify before coding

🗣️ *"Let me confirm: I need to modify nums in place so the first k elements are exactly the values not equal to val, in any order, and return k. Anything after index k doesn't matter, right?"*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Must it be in place (no new array)? | **Yes**, O(1) extra memory. | Rules out "build a filtered list". |
| Does the order of kept elements matter? | **No.** | Gives extra freedom (used in a follow-up). |
| What goes after index `k`? | Anything. It's ignored. | We don't need to clean up the tail. |
| Can the array be empty? | **Yes** (length 0). | Return 0 without crashing. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** *"While there's still a `val` in the list, delete one copy of it."*

🗣️ *"The most literal approach is to keep deleting val until it's gone, then return the length. It's very readable."*

**Why is it slow?** Two hidden costs:
1. `val in nums` **scans** the list from the start to find a copy, which is up to n steps.
2. `nums.remove(val)` deletes it, and then **every element after it shifts one place left** to close the gap. That's up to n more steps.

If there are many copies of `val`, we do this up to n times, so roughly **n × n = n²** work. With 10× more data, it can take 100× longer.

🗣️ *"Each remove is O(n) because it searches and then shifts the rest of the array. Doing that up to n times gives O(n²). The waste is that the same elements get shifted again and again."*

In [1]:
class SolutionBrute:
    def removeElement(self, nums: list[int], val: int) -> int:
        while val in nums:      # O(n) search
            nums.remove(val)    # O(n) shift of everything after it
        return len(nums)

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** we don't actually need to *delete* anything. We just need the **first `k` boxes** to hold the good values. So instead of removing the bad ones, **copy the good ones forward**.

Use **two pointers** (two index variables walking the same array):
- **Reader `r`**: walks through *every* box, one by one, looking at its value.
- **Writer `k`**: points to the **next free spot** at the front where a good value should go.

Rule: if `nums[r]` is good (≠ `val`), copy it to `nums[k]` and move `k` forward. If it's bad, do nothing and move on.

At the end, `k` is both **where the good section ends** and **how many good values there are**, which is exactly what we return.

**Why is it safe to overwrite `nums[k]`?** The writer never gets ahead of the reader (`k ≤ r` always). So we only ever overwrite boxes the reader has **already looked at**. We never lose a value we still need.

🗣️ *"Instead of deleting, I'll compact the array. I use a read pointer that scans every element and a write pointer that marks the next slot for a value I'm keeping. Whenever the read pointer finds a value that isn't val, I copy it to the write position and advance the writer. The writer never passes the reader, so I only overwrite values I've already processed. At the end, the write pointer equals k. Each element is read once and written at most once, so it's O(n) time and O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Delete in a loop (`remove`) | ❌ | Every deletion shifts the tail, so the same elements move again and again. O(n²). |
| Build a new filtered list, copy it back | ⚠️ | O(n) time, but uses a second array, which breaks the in-place rule. |
| Sort so `val`s cluster, then cut them out | ❌ | Sorting costs O(n log n) and still needs shifting. More work, not less. |
| **Read/write pointers (compaction)** | ✅ **best** | One pass, no extra memory, never shifts anything twice. |
| Swap the bad value with the **last** element and shrink the array | ✅ (alternative) | Also O(n). Fewer writes when `val` is rare, but it scrambles order. Allowed here, so it's a nice follow-up. |

**Takeaway pattern:** *"to remove things from an array in place, don't delete. Copy the keepers forward with a write pointer."* You'll reuse this in Questions 3 and 4.

In [2]:
class Solution:
    def removeElement(self, nums: list[int], val: int) -> int:
        k = 0                       # writer: next slot for a value we keep
        for r in range(len(nums)):  # reader: looks at every value once
            if nums[r] != val:
                nums[k] = nums[r]   # keep it: copy forward
                k += 1
        return k                    # count of kept values = end of the good section

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [3, 2, 2, 3]`, `val = 3`

| r (reader) | nums[r] | keep? | action | k (writer) after | nums afterwards |
|---|---|---|---|---|---|
| 0 | 3 | ❌ it's val | skip | 0 | [3, 2, 2, 3] |
| 1 | 2 | ✅ | nums[0] = 2 | 1 | [**2**, 2, 2, 3] |
| 2 | 2 | ✅ | nums[1] = 2 | 2 | [2, **2**, 2, 3] |
| 3 | 3 | ❌ it's val | skip | 2 | [2, 2, 2, 3] |

Return **k = 2**. The first two boxes are `[2, 2]` ✅. The `2, 3` after them is leftover junk the judge ignores.

**Narrated:** "My reader looks at the first box: it's a 3, which I'm removing, so I skip it. Next box is a 2, a keeper, so I write it into slot 0 and my writer moves to slot 1. Another 2 goes into slot 1, and the writer moves to 2. The last box is a 3, skip. The writer is at 2, so there are 2 keepers, and they're packed at the front."

In [3]:
def run(cls, nums, val):
    nums = nums[:]
    k = cls().removeElement(nums, val)
    return k, sorted(nums[:k])          # order of kept values doesn't matter

cases = [
    (([3, 2, 2, 3], 3), (2, [2, 2])),
    (([0, 1, 2, 2, 3, 0, 4, 2], 2), (5, [0, 0, 1, 3, 4])),
    (([], 1), (0, [])),                 # empty array
    (([1, 1, 1], 1), (0, [])),          # everything removed
    (([4, 5], 1), (2, [4, 5])),         # nothing removed
]
for args, expected in cases:
    assert run(SolutionBrute, *args) == expected
    assert run(Solution, *args) == expected
    print(args, "->", expected)
print("All tests passed ✅")

([3, 2, 2, 3], 3) -> (2, [2, 2])
([0, 1, 2, 2, 3, 0, 4, 2], 2) -> (5, [0, 0, 1, 3, 4])
([], 1) -> (0, [])
([1, 1, 1], 1) -> (0, [])
([4, 5], 1) -> (2, [4, 5])
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Delete in a loop | O(n²) | O(1) | Each deletion re-shifts the rest of the array. |
| **Read/write pointers** | **O(n)** | **O(1)** | Look at each value once; copy each keeper at most once. |

**Why O(n) time?** The `for` loop visits each index exactly once and does a constant amount of work each time (one comparison, maybe one copy). Double the array, double the work.

**Why O(1) memory?** We only store `k` and `r`, no matter how big the array is.

**Edge cases to mention:** empty array (returns 0), every element equals `val` (returns 0), no element equals `val` (returns n, and the array is unchanged).

**Likely follow-up:** *"What if `val` is rare and writes are expensive?"* → Walk with `i`. When `nums[i] == val`, overwrite it with the **last** element and shrink the length by one (don't advance `i`, because the swapped-in value still needs checking). This does fewer writes, but scrambles order, which this problem allows.

---

#### 🗣️ Full interview script (about 60 seconds)

*"I need to remove all copies of val in place and return how many elements remain, with those elements at the front.*

*The naive approach is to call remove until val is gone, but each removal shifts the rest of the array, so that's O(n²).*

*Instead, I'll compact the array with two pointers. A read pointer scans every element; a write pointer marks the next free slot for a keeper. When the reader sees a value that isn't val, I copy it to the writer's slot and advance the writer. The writer never overtakes the reader, so I only overwrite values I've already looked at. At the end, the write pointer is exactly k.*

*One pass, so O(n) time, and only two index variables, so O(1) extra space."*